In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
AQI_2015_2020 = pd.read_csv('/Users/aravindasv/Downloads/kaggle aqi/city_day.csv')

In [3]:
AQI_2015_2020.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  object 
 1   Date        29531 non-null  object 
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  object 
dtypes: float64(13), object(3)
memory usage: 3.6+ MB


In [4]:
AQI_2015_2020['City'].unique()

array(['Ahmedabad', 'Aizawl', 'Amaravati', 'Amritsar', 'Bengaluru',
       'Bhopal', 'Brajrajnagar', 'Chandigarh', 'Chennai', 'Coimbatore',
       'Delhi', 'Ernakulam', 'Gurugram', 'Guwahati', 'Hyderabad',
       'Jaipur', 'Jorapokhar', 'Kochi', 'Kolkata', 'Lucknow', 'Mumbai',
       'Patna', 'Shillong', 'Talcher', 'Thiruvananthapuram',
       'Visakhapatnam'], dtype=object)

In [5]:
"""
Cities vs States
table generated in ChatGPT

| City               | State           |
| ------------------ | --------------- |
| Ahmedabad          | Gujarat         |
| Aizawl             | Mizoram         |
| Amaravati          | Andhra Pradesh  |
| Amritsar           | Punjab          |
| Bengaluru          | Karnataka       |
| Bhopal             | Madhya Pradesh  |
| Brajrajnagar       | Odisha          |
| Chandigarh         | Chandigarh (UT) |
| Chennai            | Tamil Nadu      |
| Coimbatore         | Tamil Nadu      |
| Delhi              | Delhi (NCT)     |
| Ernakulam          | Kerala          |
| Gurugram           | Haryana         |
| Guwahati           | Assam           |
| Hyderabad          | Telangana       |
| Jaipur             | Rajasthan       |
| Jorapokhar         | Jharkhand       |
| Kochi              | Kerala          |
| Kolkata            | West Bengal     |
| Lucknow            | Uttar Pradesh   |
| Mumbai             | Maharashtra     |
| Patna              | Bihar           |
| Shillong           | Meghalaya       |
| Talcher            | Odisha          |
| Thiruvananthapuram | Kerala          |
| Visakhapatnam      | Andhra Pradesh  |

"""

'\nCities vs States\ntable generated in ChatGPT\n\n| City               | State           |\n| ------------------ | --------------- |\n| Ahmedabad          | Gujarat         |\n| Aizawl             | Mizoram         |\n| Amaravati          | Andhra Pradesh  |\n| Amritsar           | Punjab          |\n| Bengaluru          | Karnataka       |\n| Bhopal             | Madhya Pradesh  |\n| Brajrajnagar       | Odisha          |\n| Chandigarh         | Chandigarh (UT) |\n| Chennai            | Tamil Nadu      |\n| Coimbatore         | Tamil Nadu      |\n| Delhi              | Delhi (NCT)     |\n| Ernakulam          | Kerala          |\n| Gurugram           | Haryana         |\n| Guwahati           | Assam           |\n| Hyderabad          | Telangana       |\n| Jaipur             | Rajasthan       |\n| Jorapokhar         | Jharkhand       |\n| Kochi              | Kerala          |\n| Kolkata            | West Bengal     |\n| Lucknow            | Uttar Pradesh   |\n| Mumbai             | Ma

In [6]:
# extract the yearly sheets separately. AQI_2015, AQI_2017

In [7]:
AQI_2015_2020['Date'].dtype

dtype('O')

In [8]:
# modify the column to DateTime
AQI_2015_2020['Date'] = pd.to_datetime(AQI_2015_2020['Date'])
AQI_2015 = AQI_2015_2020[AQI_2015_2020['Date'].dt.year == 2015]
AQI_2015.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,0.00,0.02,0.00,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,3.68,5.50,3.77,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,6.80,16.40,2.25,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,4.43,10.14,1.00,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,7.01,18.89,2.78,NaN,NaN


In [9]:
AQI_2015.shape

(2801, 16)

In [15]:
AQI_2015.isna().any(axis=1).sum()

np.int64(2487)

In [17]:
"""
this is a massive number of rows which have nan values. but it could be because AQI, AQI_Bucket dont have the values
as we see in the first few rows
lets check what are the essential columns to calculate AQI
PM2.5, PM10, SO2, NOx, NH3, CO and O3
"""

In [19]:
# credit: https://www.kaggle.com/willkoehrsen/start-here-a-gentle-introduction. 

def missing_values_table(df):
    # Total missing values
    mis_val = df.isnull().sum()
        
    # Percentage of missing values
    mis_val_percent = 100 * df.isnull().sum() / len(df)
        
    # Make a table with the results
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
        
    # Rename the columns
    mis_val_table_ren_columns = mis_val_table.rename(
    columns = {0 : 'Missing Values', 1 : '% of Total Values'})
        
    # Sort the table by percentage of missing descending
    mis_val_table_ren_columns = mis_val_table_ren_columns[
        mis_val_table_ren_columns.iloc[:,1] != 0].sort_values(
        '% of Total Values', ascending=False).round(1)
        
    # Print some summary information
    print ("Your selected dataframe has " + str(df.shape[1]) + " columns.\n"      
        "There are " + str(mis_val_table_ren_columns.shape[0]) + " columns that have missing values.")
        
    # Return the dataframe with missing information
    return mis_val_table_ren_columns

In [20]:
missing_values_table(AQI_2015)

Your selected dataframe has 16 columns.
There are 14 columns that have missing values.


,Missing Values,% of Total Values
PM10,2289,81.7
NH3,1614,57.6
Xylene,1465,52.3
AQI,974,34.8
AQI_Bucket,974,34.8
PM2.5,945,33.7
Benzene,627,22.4
Toluene,487,17.4
SO2,484,17.3
O3,461,16.5


In [10]:
AQI_2017 = AQI_2015_2020[AQI_2015_2020['Date'].dt.year == 2017]
AQI_2017.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
731,Ahmedabad,2017-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN
732,Ahmedabad,2017-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN
733,Ahmedabad,2017-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN
734,Ahmedabad,2017-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN
735,Ahmedabad,2017-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,NaN


In [11]:
AQI_2017.shape

(4689, 16)

In [16]:
AQI_2015.isna().any(axis=1).sum()

np.int64(2487)

In [21]:
missing_values_table(AQI_2017)

Your selected dataframe has 16 columns.
There are 14 columns that have missing values.


,Missing Values,% of Total Values
Xylene,2918,62.2
PM10,2662,56.8
NH3,2587,55.2
NOx,1940,41.4
AQI,1455,31.0
AQI_Bucket,1455,31.0
PM2.5,1395,29.8
O3,1384,29.5
SO2,1364,29.1
NO2,1266,27.0
